# Retrieval-Augmented Generation: A Worked Example
### Building a RAG System over a Cybersecurity Knowledge Base

**Audience:** Graduate students who have completed the `embedding.ipynb` notebook.

**Goal:** Take the concepts (embeddings, chunking, retrieval, re-ranking) and assemble them into a complete, working RAG pipeline that answers questions from documents.

We follow the classic **six-step RAG recipe**:
1. 📄 Load & chunk the data
2. 🗄️ Connect to a vector store
3. 📥 Ingest (embed + index) the chunks
4. 🔍 Retrieve relevant context for a query
5. 🔗 Build the RAG prompt / chain
6. 💬 Ask questions with RAG

> **Reproducibility note:** This notebook runs **offline** with `sentence-transformers` and `faiss` so it works in class without any server or API key. Each step also shows the **production equivalent** (LangChain + a managed vector database like Milvus) in comments, so you can map the teaching code onto a real-world stack.

## 🛠️ Setup

In [ ]:
# If needed, install dependencies:
# !pip install -q sentence-transformers faiss-cpu numpy

import numpy as np
from sentence_transformers import SentenceTransformer

np.set_printoptions(precision=3, suppress=True)

# One embedding model, used for BOTH documents and queries.
# (Using two different models would put them in incompatible vector spaces!)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
DIM = embed_model.get_sentence_embedding_dimension()
print("Embedding model ready -", DIM, "dimensions")

# 📄 Step 1 — Load & Chunk the Data

- Read the source document from disk
- Split it into **chunks** the model can embed
- Here: split a Markdown file on its `#` section headers
- Each section becomes one retrievable chunk
- Real systems also handle PDFs, HTML, code, etc.

> ### 🎤 Instructor Script
>
> Every RAG system begins with data. We load a Markdown knowledge base and split it into chunks — the unit we will embed and retrieve. Our document is organized by `#` section headers, so splitting on `#` gives clean, self-contained passages, one per security topic. This header-based strategy is a simple form of *semantic chunking*: each chunk is already a coherent idea, which is exactly what we want for precise retrieval. Recall from the embeddings notebook why we chunk at all: embedding a whole document averages all its topics into one blurry vector. Chunking keeps each topic sharp so a query can match the specific passage that answers it.

In [ ]:
FILE_NAME = "cybersecurity_kb.md"

with open(FILE_NAME, mode="r", encoding="utf8") as f:
    text = f.read()

# Split on Markdown headers; drop empty fragments and trim whitespace.
chunks = [item.strip() for item in text.split("#") if item.strip()]

print(f"Loaded {len(chunks)} chunks from {FILE_NAME}\n")
for i, c in enumerate(chunks):
    title = c.splitlines()[0]
    print(f"  chunk {i:2d}: {title}")

# 🗄️ Step 2 — Connect to the Vector Store

- A vector store holds embeddings + their source text
- We use an in-memory **FAISS** index for teaching
- Cosine similarity via inner product on normalized vectors
- Production: a managed DB (Milvus, Pinecone, pgvector)
- Same idea — just persistent and scalable

> ### 🎤 Instructor Script
>
> Next we create the container that will hold our vectors. For class we use FAISS, an in-memory index that needs no server. We choose an inner-product index and will store normalized vectors, which makes the inner product equal to cosine similarity — the metric we want for text. In production you would instead connect to a managed vector database such as Milvus, which persists data, scales to billions of vectors, and offers fast approximate indexes like HNSW. I have left the original LangChain + Milvus connection code in comments so you can see that the conceptual step — 'create a place to store and search vectors' — is identical; only the engine changes.

In [ ]:
import faiss

# Inner-product index; with normalized vectors this equals cosine similarity.
index = faiss.IndexFlatIP(DIM)
print("Created empty FAISS index. Vectors stored:", index.ntotal)

# --- Production equivalent (the original Chinese example used Milvus) ---
# from langchain_milvus import Milvus
# from models import get_embed
# embed_fn = get_embed()
# db = Milvus(embedding_function=embed_fn,
#             auto_id=True,
#             index_params={"index_type": "HNSW", "metric_type": "L2"},
#             connection_args={"host": "localhost", "port": 19530})
# db.drop()   # clear any existing collection

# 📥 Step 3 — Ingest: Embed & Index the Chunks

- Embed every chunk into a vector
- Normalize so inner product = cosine
- Add the vectors to the index
- Keep the original text in parallel for lookup
- This is your searchable knowledge base

> ### 🎤 Instructor Script
>
> Now we populate the store. We embed all chunks in one batch, normalize them, and add them to the FAISS index; we keep the `chunks` list itself so that, given a matching vector's position, we can return the human-readable passage. This is the offline, one-time cost of RAG: embedding the corpus. After this step the index *is* our knowledge base — a matrix of meaning we can search in milliseconds. Note that we deliberately reuse `embed_model`, the same model we will use for queries. Mixing models here is a classic and silent bug: the vectors would live in different spaces and similarity scores would be meaningless.

In [ ]:
# Embed all chunks at once (normalized for cosine similarity).
chunk_vectors = embed_model.encode(chunks, normalize_embeddings=True).astype("float32")

index.add(chunk_vectors)
print("Ingested", index.ntotal, "chunks into the index.")
print("Index matrix shape:", chunk_vectors.shape, "(rows = chunks, cols = dims)")

# --- Production equivalent ---
# db.add_texts(texts=chunks)   # LangChain/Milvus embeds + stores in one call

# 🔍 Step 4 — Retrieve Relevant Context

- Embed the query with the **same** model
- Search the index for the top-k nearest chunks
- Apply a **score threshold** to drop weak matches
- Concatenate survivors into a `context` string
- This context is what we feed the LLM

> ### 🎤 Instructor Script
>
> Retrieval is the heart of RAG. We embed the query with the same model, search the index for the k nearest chunks, and keep only those above a relevance threshold. The threshold matters: if nothing is relevant, we would rather return 'no context' than feed the model junk that invites hallucination. One subtlety carried over from the embeddings notebook — thresholds are *model-specific*. The original example used 0.7–0.8 with a different model and an L2 metric; with MiniLM cosine scores, good matches often sit around 0.3–0.6, so we set the threshold accordingly. Always calibrate the threshold on your own data rather than copying a number from elsewhere.

In [ ]:
def retrieve(query, k=4):
    """Return the top-k (score, chunk) pairs for a query."""
    q = embed_model.encode(query, normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q.reshape(1, -1), k)
    return [(float(s), chunks[i]) for s, i in zip(scores[0], idx[0])]


query = "How can attackers trick employees into giving up passwords?"
print("Query:", query, "\n")
for rank, (s, chunk) in enumerate(retrieve(query, k=4), 1):
    title = chunk.splitlines()[0]
    print(f"{rank}. score={s:.3f}  ->  {title}")

Now wrap retrieval in a `get_context` helper that applies a relevance threshold (mirroring the original `get_context` function):

In [ ]:
def get_context(query, k=6, score_threshold=0.35):
    """Retrieve chunks above a relevance threshold and join them into one context block."""
    results = retrieve(query, k=k)
    kept = [chunk for score, chunk in results if score >= score_threshold]
    return "\n\n".join(kept) if kept else "No relevant context found."


query = "What does the never trust, always verify model mean?"
print("Query:", query, "\n")
print("=== RETRIEVED CONTEXT ===\n")
print(get_context(query))

# 🔗 Step 5 — Build the RAG Prompt / Chain

- A prompt template with a **system role** + the context
- Instruct the model to answer *from the context*
- Tag answers `[RAG]` (from docs) or `[LLM]` (own knowledge)
- Chain: prompt → LLM → output parser
- Context is injected fresh on every query

> ### 🎤 Instructor Script
>
> With retrieval working, we design the prompt that turns context into an answer. The system message gives the model a role, pastes in the retrieved context, and sets output rules. We borrow a nice idea from the original example: the model tags its answer `[RAG]` when it relies on the supplied context, and `[LLM]` when it falls back to its own knowledge because the context was empty or irrelevant. That tag makes grounding visible to the user and is great for debugging. In LangChain this is expressed elegantly as a chain — `prompt | llm | parser` — but conceptually it is just: fill the template, call the model, parse the text. We build a runnable, framework-free version here.

In [ ]:
SYSTEM_PROMPT = (
    "# Role\n"
    "You are a cybersecurity Q&A expert. Use the CONTEXT below to answer the user's question.\n\n"
    "# Output format\n"
    "1. If the CONTEXT is empty or does not contain the answer, answer from your own knowledge "
    "and prefix your reply with [LLM].\n"
    "2. If the answer is found in the CONTEXT, answer using it and prefix your reply with [RAG].\n\n"
    "# Context\n"
    "{context}\n"
)


def build_messages(query, context):
    """Assemble the chat messages a RAG chain would send to the LLM."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT.format(context=context)},
        {"role": "user", "content": query},
    ]


# Preview the fully assembled prompt for one query:
q = "What are the phases of incident response?"
msgs = build_messages(q, get_context(q))
print(msgs[0]["content"])
print("USER:", msgs[1]["content"])

# --- Production equivalent (LangChain) ---
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser
# from models import get_qwen_llm
# prompt = ChatPromptTemplate.from_messages([("system", SYSTEM_PROMPT), ("user", "{query}")])
# rag_chain = prompt | get_qwen_llm() | StrOutputParser()

# 💬 Step 6 — Ask Questions with RAG

- Combine: retrieve context → build prompt → generate
- Grounded answers cite your documents, not guesses
- Compare an **in-domain** vs **out-of-domain** query
- In-domain → `[RAG]`; out-of-domain → `[LLM]`
- The generation call is one API line away

> ### 🎤 Instructor Script
>
> Finally we run the full loop. For a given question we retrieve context, assemble the prompt, and send it to an LLM to generate the answer. To keep the notebook offline, the generation call is shown as a commented template you can enable with any chat model; the runnable code shows exactly what would be sent. Watch the contrast between two queries: an in-domain question about incident response pulls real passages and would be answered `[RAG]`, while an off-topic question about, say, French geography retrieves nothing relevant, so the model would fall back to `[LLM]`. That difference is the whole point of RAG — it grounds answers in your data when your data is relevant, and gets out of the way when it is not.

In [ ]:
def rag_answer(query, score_threshold=0.35):
    """Full RAG step: retrieve -> build prompt -> (generate)."""
    context = get_context(query, score_threshold=score_threshold)
    messages = build_messages(query, context)

    grounded = context != "No relevant context found."
    print("QUESTION  :", query)
    print("GROUNDED? :", "YES -> expect [RAG]" if grounded else "NO  -> expect [LLM]")
    print("CONTEXT   :", (context[:120] + "...") if grounded else context)

    # --- Real generation (uncomment and set a key to get a live answer) ---
    # from anthropic import Anthropic
    # client = Anthropic(api_key="YOUR_KEY")
    # resp = client.messages.create(
    #     model="claude-haiku-4-5-20251001", max_tokens=300,
    #     system=SYSTEM_PROMPT.format(context=context),
    #     messages=[{"role": "user", "content": query}])
    # print("ANSWER    :", resp.content[0].text)
    return messages


print("--- In-domain query ---")
_ = rag_answer("What are the phases of incident response?")

print("\n--- Out-of-domain query ---")
_ = rag_answer("What is the capital of France?")

# ✅ Summary — The RAG Recipe

- Load → **chunk** the documents
- **Embed** chunks → store in a vector index
- **Retrieve** top-k context (with a threshold)
- Inject context into a **prompt template**
- **Generate** a grounded, tagged answer

> ### 🎤 Instructor Script
>
> We assembled a complete RAG system in six steps: load and chunk the data, embed and index the chunks, retrieve relevant context with a calibrated threshold, build a prompt that injects that context, and generate a grounded answer. The same skeleton scales from this in-memory FAISS demo to a production system on Milvus or Pinecone with a hosted LLM — only the components swap out, while the pipeline stays the same. The big ideas to remember: chunk for precision, always embed queries and documents with the same model, threshold your retrieval to avoid feeding the model noise, and make grounding visible. Master this recipe and you can build a question-answering system over any document collection you are given.

In [ ]:
print("RAG pipeline:")
print("  documents -> chunk -> embed -> vector store -> retrieve(+threshold) -> prompt -> LLM -> answer")
print("\nYou just built every stage of that pipeline over a real knowledge base.")